In [4]:
import pandas as pd
import numpy as np

# Mostrar todas las columnas al imprimir DataFrames
pd.set_option('display.max_columns', None)

In [5]:
# Cargar el archivo de datos
df = pd.read_excel('../data/ultimos13a7meses.xlsx')

filas_originales = len(df)
print(f"Dimensiones originales: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\nPrimeras columnas disponibles:")
print(df.columns.tolist())

d:\DocumentsD\EntornosVirtuales\venvPy311\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Dimensiones originales: 72,345 filas × 42 columnas

Primeras columnas disponibles:
['ID_ODE', 'T_USUARIO_ALTA', 'T_USUARIO_ULT_MOD', 'ESTADO_ODE', 'FECHA_ALTA_ODE', 'CONSIGNATARIO', 'SOLICITANTE', 'C_ID_REMITO', 'FECHA_DESP', 'C_ITEM', 'ESPESOR_DESP', 'ANCHO_DESP', 'LARGO_DESP', 'PESO_NETO_DESP', 'PESO_BRUTO_DESP', 'C_POSICION_ORDEN', 'C_PEDIDO', 'MATERIAL_PADRE', 'PESO_MATERIAL_ENTRADA', 'ANCHO_PADRE', 'LARGO_PADRE', 'ESPESOR_PADRE', 'ID_MAT', 'MATERIAL', 'PESO_MATERIAL_SALIDA', 'ANCHO_HIJO', 'LARGO_HIJO', 'ESPESOR_HIJO', 'D_INT', 'DESCRIPCION_EMBALAJE', 'C_OFA_ID', 'D_UBICACION', 'NUMERO_PARTE_HIJO', 'PIEZAS_POR_PAQ_HIJO', 'PESO_MAX_DESPACHO_HIJO', 'ID_RED_HIJO', 'ARMA_TARIMA', 'FECHA_ARMADO', 'USUARIO_ARMA', 'DESARMA_TARIMA', 'FECHA_DESARME', 'USUARIO_DESARMA']


Se eliminan las filas de datos que no cuentan con valor en ARMA_TARIMA porque representan cintas a las que no se les armó tarima, por lo que no nos competen en contextos de este proyecto

In [6]:
# Eliminar cintas que no requirieron armado de tarima
df = df.dropna(subset=['ARMA_TARIMA'])

print(f"Filas después de filtrar sin ARMA_TARIMA: {len(df):,}")
print(f"Filas eliminadas: {filas_originales - len(df):,}")

Filas después de filtrar sin ARMA_TARIMA: 29,077
Filas eliminadas: 43,268


Se seleccionan variables relevantes en torno al proyecto para el modelo de predicción del desarme y para obtener las reglas del desarme (Peso máximo y número de cintas por tarima)

In [7]:
# Variables relevantes para el proyecto
variables_relevantes = [
    # Identificación de la tarima y desarme (variables centrales)
    'ARMA_TARIMA',
    'DESARMA_TARIMA',
    
    # Restricciones de armado
    'MATERIAL_PADRE',
    'CONSIGNATARIO',
    'PESO_NETO_DESP',
    
    # Dimensiones para calcular diámetro externo
    'LARGO_HIJO',
    'ESPESOR_HIJO',
    'D_INT',
    
    # Especificaciones del producto
    'NUMERO_PARTE_HIJO',
    'PIEZAS_POR_PAQ_HIJO',
    'PESO_MAX_DESPACHO_HIJO',
    
    # Tipo de embalaje (restricciones OJO HORIZONTAL / OJO VERTICAL)
    'DESCRIPCION_EMBALAJE',
    
    # Planta (afecta límite de ancho en OJO HORIZONTAL)
    'D_UBICACION',
    
    # Ancho para verificar homogeneidad de NUMERO_PARTE_HIJO
    'ANCHO_HIJO',
]

df = df[variables_relevantes]

print(f"Variables conservadas: {len(df.columns)}")
print(f"Dimensiones del DataFrame limpio: {df.shape[0]:,} filas × {df.shape[1]} columnas")

Variables conservadas: 14
Dimensiones del DataFrame limpio: 29,077 filas × 14 columnas


Volvemos binaria la variable DESARMA_TARIMA para identificar más fácil cuando una tarima se desarma o no

0 - Es que la tarima no se desarmó

1 - La tarima se desarmó

In [8]:
# Transformar DESARMA_TARIMA a binaria: tiene valor → 1, faltante → 0
df['DESARMA_TARIMA'] = df['DESARMA_TARIMA'].notna().astype(int)

print("Distribución de DESARMA_TARIMA:")
print(df['DESARMA_TARIMA'].value_counts())
print(f"\nPorcentaje de tarimas desarmadas: {df['DESARMA_TARIMA'].mean()*100:.1f}%")

Distribución de DESARMA_TARIMA:
DESARMA_TARIMA
0    23437
1     5640
Name: count, dtype: int64

Porcentaje de tarimas desarmadas: 19.4%


En las variables que representarán las reglas del armado de tarima (cuántas cintas deben de ir por tarima y el peso máximo) se les cambia el valor "NOA" por un valor nulo para determinar de mejor manera cuando Ternium no tiene el valor de cuántas cintas deben de ir por tarima o cuánto es el peso máximo que pueden llevar dichas tarimas

In [9]:
# Reemplazar 'NOA' por NaN en las columnas de especificaciones
df['PIEZAS_POR_PAQ_HIJO'] = df['PIEZAS_POR_PAQ_HIJO'].replace('NOA', np.nan)
df['PESO_MAX_DESPACHO_HIJO'] = df['PESO_MAX_DESPACHO_HIJO'].replace('NOA', np.nan)

# Convertir a numérico ahora que NOA ya no interfiere
df['PIEZAS_POR_PAQ_HIJO'] = pd.to_numeric(df['PIEZAS_POR_PAQ_HIJO'], errors='coerce')
df['PESO_MAX_DESPACHO_HIJO'] = pd.to_numeric(df['PESO_MAX_DESPACHO_HIJO'], errors='coerce')

print("Valores faltantes por columna:")
print(df[['PIEZAS_POR_PAQ_HIJO', 'PESO_MAX_DESPACHO_HIJO']].isna().sum())
print(f"\nPorcentaje NOA en PIEZAS_POR_PAQ_HIJO:   {df['PIEZAS_POR_PAQ_HIJO'].isna().mean()*100:.1f}%")
print(f"Porcentaje NOA en PESO_MAX_DESPACHO_HIJO: {df['PESO_MAX_DESPACHO_HIJO'].isna().mean()*100:.1f}%")

Valores faltantes por columna:
PIEZAS_POR_PAQ_HIJO       25183
PESO_MAX_DESPACHO_HIJO    21152
dtype: int64

Porcentaje NOA en PIEZAS_POR_PAQ_HIJO:   86.6%
Porcentaje NOA en PESO_MAX_DESPACHO_HIJO: 72.7%


Debido a que existen varios datos faltantes en la variable 'D_INT', se busca inferir estos datos en base al NUMERO_PARTE_HIJO.
Se determina si un cierto NUMERO_PARTE_HIJO siempre usó el mismo 'D_INT', si es el caso, se considera este numero de parte como consistente y a los cintas que no tengan este dato y tengan este tipo de número de parte se le imputa dicho valor del diametro interno.
Por otro lado, si las cintas que pertenecen el numero de parte tienen varios valores de diametros internos asignados se considera este numero de parte como un numero de parte mixto, y aquellas cintas que no cuenten con el valor del diametro interno y tengan un numero de parte mixto, se eliminarán para evitar ruido y dejar valores faltantes.

In [10]:
# Convertir D_INT a numérico
df['D_INT'] = pd.to_numeric(df['D_INT'], errors='coerce')

# Clasificar números de parte según cuántos valores distintos de D_INT tienen
valores_por_parte = (
    df[df['D_INT'].notna()]
    .groupby('NUMERO_PARTE_HIJO')['D_INT']
    .nunique()
)
partes_consistentes = valores_por_parte[valores_por_parte == 1].index
partes_mixtas       = valores_por_parte[valores_por_parte > 1].index

print(f"Números de parte con D_INT consistente (1 valor): {len(partes_consistentes)}")
print(f"Números de parte con D_INT en mezcla (>1 valor):  {len(partes_mixtas)}")
print(f"  → Partes con mezcla: {list(partes_mixtas)}")

# Eliminar filas de partes con mezcla que tienen D_INT faltante
mascara_eliminar = df['NUMERO_PARTE_HIJO'].isin(partes_mixtas) & df['D_INT'].isna()
filas_eliminadas = mascara_eliminar.sum()
df = df[~mascara_eliminar]

print(f"\nFilas eliminadas por D_INT faltante en partes con mezcla: {filas_eliminadas}")

# Imputar D_INT para partes consistentes usando el único valor que tienen
mapa_d_int = (
    df[df['D_INT'].notna()]
    .groupby('NUMERO_PARTE_HIJO')['D_INT']
    .first()
)
filas_imputadas = df['D_INT'].isna().sum()
df['D_INT'] = df['D_INT'].fillna(df['NUMERO_PARTE_HIJO'].map(mapa_d_int))

print(f"Filas imputadas con D_INT del mismo número de parte: {filas_imputadas}")
print(f"Filas con D_INT faltante restante: {df['D_INT'].isna().sum()}")

Números de parte con D_INT consistente (1 valor): 646
Números de parte con D_INT en mezcla (>1 valor):  5
  → Partes con mezcla: ['180580AA', '2029804C', '2029843E', '2032499I', '3.61']

Filas eliminadas por D_INT faltante en partes con mezcla: 107
Filas imputadas con D_INT del mismo número de parte: 6100
Filas con D_INT faltante restante: 822


Se calcula el diametro externo de cada cinta y nos aseguramos que los valores con los que se obtiene este valor sean numéricos. Se crea una variable en el dataset para el diametro externo de cada cinta

In [11]:
# Convertir dimensiones a numérico
df['LARGO_HIJO'] = pd.to_numeric(df['LARGO_HIJO'], errors='coerce')
df['ESPESOR_HIJO'] = pd.to_numeric(df['ESPESOR_HIJO'], errors='coerce')

# LARGO_HIJO está en metros — convertir a mm para la fórmula
LARGO_HIJO_MM = df['LARGO_HIJO'] * 1000


# Calcular diámetro externo: D_externo = √(D_INT² + 4 × LARGO_mm × ESPESOR_mm / π)
df['D_EXTERNO'] = np.sqrt(df['D_INT']**2 + (4 * LARGO_HIJO_MM * df['ESPESOR_HIJO']) / np.pi)

print("Estadísticas de D_EXTERNO (mm):")
print(df['D_EXTERNO'].describe().round(2))
print(f"\nValores nulos en D_EXTERNO: {df['D_EXTERNO'].isna().sum()}")

Estadísticas de D_EXTERNO (mm):
count    28136.00
mean      1335.97
std        289.15
min        508.00
25%       1085.61
50%       1311.05
75%       1647.30
max       4356.91
Name: D_EXTERNO, dtype: float64

Valores nulos en D_EXTERNO: 834


Se eliminan las cintas que no tienen valor asignado en largo o en espesor para evitar problemas

In [12]:
# Eliminar filas con D_EXTERNO nulo (LARGO_HIJO o ESPESOR_HIJO faltantes)
filas_antes = len(df)
df = df.dropna(subset=['D_EXTERNO'])

print(f"Filas eliminadas por D_EXTERNO nulo: {filas_antes - len(df)}")
print(f"Filas restantes: {len(df):,}")

Filas eliminadas por D_EXTERNO nulo: 834
Filas restantes: 28,136


Se eliminan las instancias que no cuenten con numero de parte hijo para evitar problemas con el modelado 

In [13]:
# Eliminar filas sin NUMERO_PARTE_HIJO
filas_antes = len(df)
df = df.dropna(subset=['NUMERO_PARTE_HIJO'])

print(f"Filas eliminadas por NUMERO_PARTE_HIJO nulo: {filas_antes - len(df)}")
print(f"Filas restantes: {len(df):,}")

Filas eliminadas por NUMERO_PARTE_HIJO nulo: 72
Filas restantes: 28,064


Se eliminan los casos en los que se juntan varias cintas de diferente material padre en una tarima y estas no se desarmaron. Como mencionó Ternium, existen casos muy especificos donde se arman tarimas con cintas de diferente material padre, como cuando una alguna cinta queda sola. Para fines de este proyecto se eliminan estos casos para evitar que el modelo se confunda y tome en cuenta la posibilidad de juntar cintas de diferentes materiales padre en una tarima. Esto no es lo ideal

In [14]:
# Identificar tarimas con diferente MATERIAL_PADRE
tarimas_multi_padre = (
    df.groupby('ARMA_TARIMA')['MATERIAL_PADRE']
    .nunique()
)
tarimas_multi_padre_idx = tarimas_multi_padre[tarimas_multi_padre > 1].index

# De esas, eliminar solo las que NO se desarmaron (excepciones legítimas no modelables)
tarimas_multi_no_desarmadas = (
    df[df['ARMA_TARIMA'].isin(tarimas_multi_padre_idx)]
    .groupby('ARMA_TARIMA')['DESARMA_TARIMA']
    .max()
)
tarimas_a_eliminar = tarimas_multi_no_desarmadas[tarimas_multi_no_desarmadas == 0].index

filas_antes = len(df)
df = df[~df['ARMA_TARIMA'].isin(tarimas_a_eliminar)]

print(f"Tarimas con diferente MATERIAL_PADRE: {len(tarimas_multi_padre_idx)}")
print(f"  - No desarmadas (eliminadas): {len(tarimas_a_eliminar)}")
print(f"  - Desarmadas (conservadas):   {len(tarimas_multi_padre_idx) - len(tarimas_a_eliminar)}")
print(f"\nFilas eliminadas: {filas_antes - len(df)}")
print(f"Filas restantes: {len(df):,}")

Tarimas con diferente MATERIAL_PADRE: 157
  - No desarmadas (eliminadas): 107
  - Desarmadas (conservadas):   50

Filas eliminadas: 270
Filas restantes: 27,794


Se eliminan todos los datos que tienen un tipo de embalaje no especificado por ternium (todo lo que no sea OJO VERTICAL u OJO HORIZONTAL) esto porque no tenemos ninguna especificaión de cómo deberían de irse estas tarimas

In [15]:
# Eliminar tarimas con tipo de embalaje no documentado
def clasificar_embalaje(descripcion):
    if pd.isna(descripcion):
        return 'DESCONOCIDO'
    descripcion = str(descripcion).upper()
    if 'OJO HORIZONTAL' in descripcion:
        return 'OJO_HORIZONTAL'
    elif 'OJO VERTICAL' in descripcion:
        return 'OJO_VERTICAL'
    else:
        return 'OTRO'

df['TIPO_EMBALAJE'] = df['DESCRIPCION_EMBALAJE'].apply(clasificar_embalaje)

tarimas_otro = df[df['TIPO_EMBALAJE'] == 'OTRO']['ARMA_TARIMA'].unique()

filas_antes = len(df)
df = df[~df['ARMA_TARIMA'].isin(tarimas_otro)]

print(f"Tarimas eliminadas por embalaje no documentado: {len(tarimas_otro)}")
print(f"Filas eliminadas: {filas_antes - len(df)}")
print(f"Filas restantes: {len(df):,}")

Tarimas eliminadas por embalaje no documentado: 0
Filas eliminadas: 0
Filas restantes: 27,794


In [16]:
# Guardar el DataFrame limpio en outputs/
df.to_csv('../outputs/01_datos_limpios.csv', index=False)

print("Archivo guardado: outputs/01_datos_limpios.csv")
print(f"Dimensiones finales: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\nResumen de variables en el archivo limpio:")
print(df.dtypes)

Archivo guardado: outputs/01_datos_limpios.csv
Dimensiones finales: 27,794 filas × 16 columnas

Resumen de variables en el archivo limpio:
ARMA_TARIMA                   str
DESARMA_TARIMA              int64
MATERIAL_PADRE                str
CONSIGNATARIO                 str
PESO_NETO_DESP            float64
LARGO_HIJO                float64
ESPESOR_HIJO              float64
D_INT                     float64
NUMERO_PARTE_HIJO             str
PIEZAS_POR_PAQ_HIJO       float64
PESO_MAX_DESPACHO_HIJO    float64
DESCRIPCION_EMBALAJE          str
D_UBICACION                   str
ANCHO_HIJO                float64
D_EXTERNO                 float64
TIPO_EMBALAJE                 str
dtype: object
